In [1]:
import pandas as pd
import pyreadstat

FILE = r"C:/Users/Lenovo/London_Sport_2/data/raw/UKDA-8223-spss/spss/spss28/active_lives_survey_nov_15-16_data_year_1_shared_20250106.sav"   # <-- your real path, raw string

# Full read once, so you can search the dictionary. Slow, but you only do it here.
df_full, meta = pyreadstat.read_sav(FILE)
print(df_full.shape)   # expect ~ (180000, 3165)

(198911, 3165)


In [28]:
# Two search tools for finding the variable you need among thousands.
# find_vars      -> searches the human-readable LABELS (the question text)
# find_var_names -> searches the variable NAMES themselves
# Both are needed because some variables (e.g. wt_final, LA) are only findable
# by name, while others are only obvious from their label.
def find_vars(meta, *keywords):           # search the LABELS
    return {v: l for v, l in meta.column_names_to_labels.items()
            if l and any(k.lower() in l.lower() for k in keywords)}

def find_var_names(meta, *keywords):      # search the NAMES
    return [v for v in meta.column_names_to_labels
            if any(k.lower() in v.lower() for k in keywords)]

In [29]:
# The ONLY thing that changes between survey waves. Each wave gets one entry
# mapping our standard concept names (weight, la_code, ...) to that wave's
# actual variable names. The code below never changes — only this dict does.
# When you add 2016_17, 2017_18, etc., look up their variable names with the
# find_vars helpers and add a new block here.
WAVE_CONFIG = {
    "2015_16": {
        "file": FILE,
        "weight": "wt_final",                       # main full-sample survey weight
        "la_code": "LA",                            # local authority (numeric; labels embed E09 GSS code)
        "activity_band": "MEMS7GR_SPORTCOUNT_A01",   # 0=Inactive 1=Insufficiently Active 2=Active; matches SE ~61%
        "age": "Age5",                              # age in 5 bands
        "gender": "Gend3",
        "ethnicity": "Eth7",                        # ethnicity in 7 groups
        "disability": "Disab3",                     # whether limiting disability
    },
}

In [30]:
# Keep only respondents in the 33 Greater London local authorities.
# LA is stored as numbers (1.0, 2.0, ...) whose value-LABELS embed the ONS GSS
# code, e.g. 9.0 -> 'E09000002 Barking and Dagenham'. All London codes start
# 'E09', so we find every numeric code whose label begins 'E09' and keep those
# rows. Keying off the GSS prefix (not borough names) avoids spelling mismatches.
def london_filter(df, meta, cfg):
    la = cfg["la_code"]
    labels = meta.variable_value_labels.get(la, {})   # {code: 'E09... Borough'}
    london_codes = [code for code, label in labels.items()
                    if str(label).startswith("E09")]  # the 33 London LAs
    if not london_codes:
        raise ValueError(f"No London codes found in value labels for {la!r}")
    return df[df[la].isin(london_codes)].copy()

In [31]:
# Turn raw survey columns into analysis-ready ones:
#  1) Replace the survey's missing-data codes (-90 to -99, e.g. "prefer not to
#     say", "not applicable") with real NaN so they don't get counted as data.
#     NOTE: we deliberately do NOT strip -1, which is a genuine value elsewhere.
#  2) Decode the coded columns we care about into readable labels using the
#     metadata dictionaries (e.g. 2 -> "Active", 1 -> "Female").
#  3) Force the weight to numeric, and copy the age band across.
# Uses cfg.get(key) so it silently skips any concept missing from the config —
# that's what lets the national sanity-check cells pass a trimmed-down config.
def clean_wave(df, meta, cfg, missing_codes=tuple(range(-99, -89))):  # -99..-90
    df = df.copy()
    df[df.isin(list(missing_codes))] = pd.NA          # missing codes -> NaN

    for key in ["activity_band", "gender", "ethnicity", "disability"]:
        var = cfg.get(key)
        if var and var in meta.variable_value_labels:
            df[key] = df[var].map(meta.variable_value_labels[var])  # decode to labels
        elif var:
            df[key] = df[var]

    df["weight"] = pd.to_numeric(df[cfg["weight"]], errors="coerce")
    if cfg.get("age"):
        df["age"] = df[cfg["age"]]   # Age5 is already a band — keep as-is
    return df

In [32]:
# Every population figure must use the survey weight, never a raw row count.
# A weighted share = (sum of weights where the condition is true)
#                    / (total weight in the group).
# If group_cols is given, it returns one row per group plus 'n', the unweighted
# sample size — use n to spot groups too small to trust (flag anything under ~30-50).
def weighted_share(df, group_cols, target_col, target_value):
    d = df.dropna(subset=["weight", target_col]).copy()
    d["_num"] = (d[target_col] == target_value).astype(float) * d["weight"]
    if group_cols:
        g = d.groupby(group_cols, dropna=False)
        out = (g["_num"].sum() / g["weight"].sum()).rename("weighted_share").reset_index()
        out["n"] = g.size().values          # unweighted base
        return out
    return d["_num"].sum() / d["weight"].sum()   # single overall figure

In [33]:
# One function that turns a raw wave into a clean, London-only DataFrame:
#   read only the columns we need -> filter to London -> clean/decode -> tag wave.
# usecols keeps memory down by returning only the configured variables.
# Returns the cleaned data plus its metadata. To process another wave later,
# just call process_wave("2016_17", WAVE_CONFIG) once that wave is in the config.
def process_wave(wave_key, config):
    cfg = config[wave_key]
    wanted = [cfg[k] for k in ["weight","la_code","activity_band","age",
                               "gender","ethnicity","disability"] if cfg.get(k)]
    df, meta = pyreadstat.read_sav(cfg["file"], usecols=wanted)
    df = london_filter(df, meta, cfg)
    df = clean_wave(df, meta, cfg)
    df["wave"] = wave_key      # so waves stay identifiable once stacked together
    return df, meta

In [34]:
%%time

# Execute the pipeline for 2015-16 and save the clean London data to parquet.
# parquet is a fast, compact format — later analysis can reload this instantly
# instead of reprocessing from the raw .sav. %%time prints how long it took.

london_2015_16, meta_2015_16 = process_wave("2015_16", WAVE_CONFIG)
london_2015_16.to_parquet("london_2015_16_clean.parquet")
print(london_2015_16.shape)   # London rows x columns

(19887, 14)
CPU times: total: 15 s
Wall time: 15.5 s


In [35]:
# Validation step: does our decoding + weighting reproduce Sport England's
# published national figure (~61% active for 2015-16)? We clean ONLY the
# activity band + weight from the full national sample (fast, 2 columns) using
# a trimmed config so clean_wave doesn't look for demographic columns we left out.
base = WAVE_CONFIG["2015_16"]
cfg = {"activity_band": base["activity_band"], "weight": base["weight"]}
nat_small = df_full[[cfg["activity_band"], cfg["weight"]]].copy()
nat = clean_wave(nat_small, meta, cfg)
print(weighted_share(nat, [], "activity_band", "Active"))

0.6207443376518277


In [36]:
# Active Lives has several "activity level" variables (different official
# definitions). We don't know which one matches the published headline, so we
# compute the national weighted % Active for each candidate. Whichever lands
# nearest ~0.61 is the definition Sport England reports — set that as
# 'activity_band' in WAVE_CONFIG. (MEMS7GR_ALL is coded as INactivity and may
# label its categories differently — check its value labels if it's the match.)
base = WAVE_CONFIG["2015_16"]
for v in ["MEMS7GR_SPORTFUND_A02", "MEMS7GR_SPORTCOUNT_A01",
          "MEMS7GR_SPORTPRE2016_A03", "MEMS7GR_ALL"]:
    cfg = {"activity_band": v, "weight": base["weight"]}
    small = df_full[[v, base["weight"]]].copy()
    n = clean_wave(small, meta, cfg)
    print(v, round(weighted_share(n, [], "activity_band", "Active"), 3))

MEMS7GR_SPORTFUND_A02 0.546
MEMS7GR_SPORTCOUNT_A01 0.621
MEMS7GR_SPORTPRE2016_A03 0.402
MEMS7GR_ALL 0.654


In [ ]:
# The actual outputs for the project: London's overall weighted % Active, then
# broken down by ethnicity (uncomment the others for age and gender). Each
# breakdown row carries 'n' so you can see which categories have enough sample
# to be reliable. The NaN ethnicity row = respondents with no ethnicity recorded;
# check london_2015_16["ethnicity"].isna().mean() to see how big that group is.
print("Overall London % Active:",
      round(weighted_share(london_2015_16, [], "activity_band", "Active"), 3))

weighted_share(london_2015_16, ["ethnicity"], "activity_band", "Active")
# weighted_share(london_2015_16, ["age"], "activity_band", "Active")
# weighted_share(london_2015_16, ["gender"], "activity_band", "Active")

## Wave 1 (2015–16) — Findings

### What this notebook does
Takes the raw Active Lives 2015–16 SPSS file (198,911 respondents, 3,165 variables),
filters to the 33 Greater London local authorities, decodes and cleans the variables
of interest, and produces **survey-weighted** activity figures overall and by ethnicity.
The logic is wave-agnostic: only the `WAVE_CONFIG` entry changes when adding later waves.

### Key decisions
- **Activity measure:** `MEMS7GR_SPORTCOUNT_A01` (Sport England "count" definition;
  0 = Inactive, 1 = Insufficiently Active, 2 = Active). Chosen because its national
  weighted active rate (**62.1%**) reproduces Sport England's published 2015–16
  headline (~61%), confirming the decoding and weighting are correct. The alternative
  "fund" definition (`SPORTFUND`, 54.6%) and the pre-2016 definition (40.2%) were
  rejected as they do not match the published headline.
- **Weighting:** all percentages use the survey weight `wt_final`; no figure is a raw
  row count.
- **Missing data:** survey codes −90 to −99 recoded to missing before analysis.
- **Geography:** London identified via the ONS GSS code prefix `E09` embedded in the
  local-authority value labels, not borough names (avoids spelling mismatches).

### Headline results — Greater London, 2015–16
- **Overall active rate: 63.5%** (weighted), based on 19,887 London respondents.
- This sits slightly **above** the national figure of 62.1% — plausibly reflecting
  London's younger age profile, but worth noting against the expectation that London
  has historically tracked around the England average.

### Activity by ethnicity (weighted % active; n = unweighted base)
| Ethnicity | % Active | n |
|---|---|---|
| Mixed | 73.4% | 522 |
| White Other | 68.9% | 2,855 |
| White British | 67.1% | 10,782 |
| Chinese | 61.1% | 319 |
| Black | 57.7% | 1,297 |
| Other ethnic group | 55.6% | 529 |
| South Asian | 54.7% | 2,365 |
| (Not recorded) | 59.3% | 1,218 |

A clear gradient: White and Mixed groups are most active; South Asian, "Other", and
Black groups least active — consistent with established national patterns.

### Caveats
- **Small bases:** Chinese (n=319) and Mixed (n=522) rest on smaller samples, so those
  estimates are indicative rather than precise.
- **Missing ethnicity:** 1,218 respondents (~6% of the London sample) have no recorded
  ethnicity; shown as a separate row rather than dropped, for transparency.
- These are **repeated cross-sections**, not the same individuals over time — suitable
  for tracking population trends, not individual change.

### Next steps
- Add the remaining waves (2016–17 … 2023–24) as new `WAVE_CONFIG` entries; the
  pipeline code is unchanged.
- **Harmonise** categories across waves (activity definition, ethnicity, age bands)
  via a mapping table so they are comparable year-on-year.
- Stack all waves into a single London panel for trend analysis and forecasting.